# Phase 8: generation runs on Colab

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save.

Run the cells **in order**, one at a time (▶ or Shift+Enter), and wait for the green ✓ before the next one.
Needs `rag_data.zip` (`data/raw`, `data/processed`, `data/chroma`) in the top level of My Drive.

| Part | Cells | When |
|---|---|---|
| Tune run (Step 1) | 1 → 2 → 3 → 4 → 5 → 6 → 7 | now |
| Final test run (Step 5) | 1 → 2 → 8 | only after Steps 2-4 are done and `configs/gen_final.yaml` is pushed |

**If the runtime disconnects:** run cells 1 and 2 again, then go straight back to the cell that was running (5 or 8).
Answers are saved to My Drive as they are made, so it continues where it stopped.

Output is kept short on purpose (long progress bars slow the browser tab down).

## 1. Drive, code, data, packages (about 3-4 min)
A pop-up asks for Drive access: **Connect to Google Drive → Allow**. Expect the latest commit line, then `Index chunks: 2468 (expect 2468)`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/rag
!git clone -q -b phase-8-generation https://github.com/himalSubedi0602/Rag_System.git /content/rag
%cd /content/rag
!git log --oneline -1
!unzip -q -o /content/drive/MyDrive/rag_data.zip -d /content/rag
!pip install -q -r requirements.txt 2>&1 | grep -v -i "dependency" | tail -3
import chromadb
n = chromadb.PersistentClient('data/chroma').get_collection('fixed-500-50__bge-small__noprefix').count()
print('Index chunks:', n, '(expect 2468)')
assert n == 2468, 'Wrong or missing index: rag_data.zip must include data/chroma'

## 2. Ollama and both models (about 4-6 min)
Each pull prints one line (`success`); the list shows `llama3.1:8b` and `qwen2.5:7b`.

In [ ]:
!sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh 2>/dev/null | sh 2>&1 | tail -2
import subprocess, time, urllib.request
subprocess.Popen(['ollama', 'serve'], stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://localhost:11434/api/version', timeout=2); break
    except Exception:
        time.sleep(1)
!ollama pull llama3.1:8b 2>&1 | tail -1
!ollama pull qwen2.5:7b 2>&1 | tail -1
!ollama list

## 3. Tests and a 5-question smoke run (about 2 min)
GPU for embeddings (`device: auto`; `cpu` is only for the 8 GB Mac). Expect `147 passed` or more, then 5 answers of a few seconds each. Written to `gen_llama_smoke`: not a result.

If any answer takes more than 60 s, stop and report it.

In [ ]:
%cd /content/rag
!sed -i 's/^device: cpu/device: auto/' configs/gen_llama.yaml
!grep '^device' configs/gen_llama.yaml
!python -m pytest -q 2>&1 | tail -1
!python run_generation.py configs/gen_llama.yaml --limit 5

## 4. Clear leftovers from earlier attempts (safe to run any time)
Old files (from runs before the reply cap) are **moved** to `gen_llama_old_<time>`, never deleted. Progress made by the current code is never touched.

In [ ]:
import os, shutil, time, yaml
d = '/content/drive/MyDrive/gen_llama'
gen, cfg_path = f'{d}/generations.jsonl', f'{d}/config.yaml'
if not os.path.exists(d):
    print('No old folder: nothing to clean')
elif os.path.exists(gen) and os.path.exists(cfg_path) and \
        (yaml.safe_load(open(cfg_path)) or {}).get('generation', {}).get('max_tokens') == 512:
    n = sum(1 for line in open(gen) if line.strip())
    print(f'Not cleaning: {n} answers here come from the current code (your resume progress). Go to cell 5.')
else:
    old = f'{d}_old_{time.strftime("%Y%m%d_%H%M")}'
    shutil.move(d, old)
    print('Moved old leftovers to', old)

## 5. Full tune run, saved to Drive as it goes (about 20-30 min)
159 questions, then grading with the qwen2.5:7b judge. Keep this tab open and in front. Prints `Resuming: N answers ...` after a disconnect.

In [ ]:
%cd /content/rag
!sed -i 's/^device: cpu/device: auto/' configs/gen_llama.yaml
!mkdir -p /content/drive/MyDrive/gen_llama
!rm -rf eval/results/gen_llama && ln -s /content/drive/MyDrive/gen_llama eval/results/gen_llama
!python run_generation.py configs/gen_llama.yaml
!ls /content/drive/MyDrive/gen_llama

## 6. Zip the results and download
If the download doesn't start, get `gen_llama.zip` from the top level of My Drive.

In [ ]:
!cd /content/drive/MyDrive && rm -f gen_llama.zip && zip -r -q gen_llama.zip gen_llama && ls -lh gen_llama.zip
from google.colab import files
files.download('/content/drive/MyDrive/gen_llama.zip')

## 7. Fallback trigger: report only (pre-registered)
Prints whether citation-following is poor (format < 0.90 or support < 0.70). It does **not** run the fallback: report the result first.

In [ ]:
import pandas as pd
m = pd.read_csv('/content/drive/MyDrive/gen_llama/metrics.csv', index_col=0).loc['overall']
fmt, sup = m['cit_format'], m['cit_supported']
print(f'Citation format {fmt:.3f} (trigger < 0.90), support {sup:.3f} (trigger < 0.70)')
print('FALLBACK TRIGGER FIRED: report it before running anything.' if fmt < 0.90 or sup < 0.70
      else 'No fallback trigger: llama3.1:8b stays.')

## 8. FINAL test run (Step 5): run ONCE, only after Steps 2-4
Needs `configs/gen_final.yaml` pushed to the branch (made on the laptop by `python -m src.final_config gen_llama --write`).
Run cells 1 and 2 first, then this one. It refuses to start if the final run is already complete, and resumes
if it was cut off (still the same single run). Results go to `My Drive/gen_final` and are zipped for download.

In [ ]:
%cd /content/rag
!git pull -q
import json, os
assert os.path.exists('configs/gen_final.yaml'), 'configs/gen_final.yaml is missing: make, commit and push it first'
!sed -i 's/^device: cpu/device: auto/' configs/gen_final.yaml
!grep -E '^(run_name|device)|split|decline_threshold|  model' configs/gen_final.yaml
d = '/content/drive/MyDrive/gen_final'
os.makedirs(d, exist_ok=True)
n_test = sum(json.loads(line)['split'] == 'test' for line in open('eval/questions.jsonl'))
gen = f'{d}/generations.jsonl'
n_done = sum(1 for line in open(gen) if line.strip()) if os.path.exists(gen) else 0
if n_done >= n_test and os.path.exists(f'{d}/summary.md'):
    raise SystemExit(f'The final test run is already complete ({n_done}/{n_test}): it runs only once. Nothing done.')
print(f'Resuming the final run: {n_done}/{n_test} answers saved' if n_done else f'Starting the one final run ({n_test} questions)')
!rm -rf eval/results/gen_final && ln -s {d} eval/results/gen_final
!python run_generation.py configs/gen_final.yaml --final
!cd /content/drive/MyDrive && rm -f gen_final.zip && zip -r -q gen_final.zip gen_final && ls -lh gen_final.zip
from google.colab import files
files.download('/content/drive/MyDrive/gen_final.zip')